# Notebook 08: fronteiras, ordinalidade e calibração

**Unidade curricular:** Dados e Aprendizagem Automática (DAA), 2026/2027

O diagnóstico dos notebooks anteriores mostrou dois problemas. Cerca de 75% dos erros ficam na faixa vizinha e 41% das previsões com probabilidade igual ou superior a 0.6 estão erradas. Este notebook testa métodos que atacam a localização da fronteira e a confiança do modelo.

O protocolo é o do Notebook 07: cinco folds estratificadas com seed 42, XGBoost com 500 árvores, profundidade 6, learning rate 0.05 e pesos balanceados, e Macro-F1 por fold. A referência é o XGBoost multiclasse já guardado, com 0.4740.

Cada experiência tem um bloco de texto com a pergunta, o método e o que se espera, seguido do código e de uma tabela com a média, o desvio-padrão, a diferença para a referência e em quantas folds ganhou. Uma variante só é adotada se ganhar pelo menos 0.005 e vencer em quatro das cinco folds.

As experiências vão da decomposição cumulativa a alternativas de boosting, como LightGBM e CatBoost, passando por calibração, meta-stacking e uma rede neuronal. Os resultados ficam em `data/nb08_*.json` e `data/nb08_*.npz`. Apagar um destes ficheiros força a reexecução da experiência correspondente. Nada é submetido ao Kaggle a partir do notebook.


In [1]:
import os
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBClassifier
import xgboost as xgb
from scipy.optimize import minimize_scalar, minimize

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SPLITS = 5
K = 16
CLASSES = np.arange(K)
PARES = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8),
         (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15)]


def _data_dir():
    for cand in [os.path.join("..", "..", "data"), os.path.join("..", "data"), "data"]:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError("pasta data/ nao encontrada")


DATA_DIR = _data_dir()
TARGET = "AGE_RANGE"

# MODO="rapido" (screening): dados completos + 2 folds, para smoke tests em ~5 min.
# NAO usar subsample: 50% + 2 folds deixaria so 25% para treino e distorcia a
# comparação (o F&H perdia o ganho todo). A decisão é sempre no modo "completo".
MODO = os.environ.get("NB08_MODO", "completo")
SUF = "_rap" if MODO in ("rapido", "ecra") else ""

train = pd.read_csv(os.path.join(DATA_DIR, "train_nb02.csv"))
y = train[TARGET].to_numpy().astype(int)
X = train.drop(columns=["PID", TARGET])
Xv = X.to_numpy()

SUB = None
if MODO in ("rapido", "ecra"):
    N_SPLITS = 2

FOLDS = list(
    StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, y)
)
FREQ = np.bincount(y, minlength=K) / len(y)
SW_BAL = compute_sample_weight("balanced", y)
print(f"MODO={MODO} | n={len(y)} | folds={len(FOLDS)}")


def gravar(nome, obj):
    with open(os.path.join(DATA_DIR, f"nb08_{nome}{SUF}.json"), "w") as f:
        json.dump(obj, f, indent=1, default=float)


def carregar(nome):
    p = os.path.join(DATA_DIR, f"nb08_{nome}{SUF}.json")
    return json.load(open(p)) if os.path.exists(p) else None


RESULTADOS = []


def registar(exp, variante, f1s, base_f1s, nota=""):
    f1s = np.asarray(f1s, dtype=float)
    base_f1s = np.asarray(base_f1s, dtype=float)
    r = {
        "exp": exp,
        "variante": variante,
        "f1": float(f1s.mean()),
        "std": float(f1s.std()),
        "delta": float(f1s.mean() - base_f1s.mean()),
        "folds_ganhos": int((f1s > base_f1s).sum()),
        "nota": nota,
    }
    RESULTADOS.append(r)
    return r


def tabela(rows):
    df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    df["f1"] = df["f1"].round(4)
    df["std"] = df["std"].round(4)
    df["delta"] = df["delta"].round(4)
    return df

MODO=completo | n=10962 | folds=5


In [2]:
# --- Base: OOF do XGB multiclasse (mesmas folds) -----------------------------
d = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))
base_pred, base_proba = d["oof"].astype(int), d["proba"].astype(float)
if SUB is not None:
    base_pred, base_proba = base_pred[SUB], base_proba[SUB]
base_f1s = np.array(
    [f1_score(y[va], base_pred[va], average="macro") for _, va in FOLDS]
)
BASE = float(base_f1s.mean())
print(f"BASE XGB multiclasse: {BASE:.4f} +/- {base_f1s.std():.4f}")
print("F1 por fold:", np.round(base_f1s, 4).tolist())

BASE XGB multiclasse: 0.4736 +/- 0.0062
F1 por fold: [0.4813, 0.4796, 0.4645, 0.473, 0.4695]


## Decomposição cumulativa (Frank & Hall)

Em vez de um classificador com 16 saídas, treinamos 15 classificadores binários, um por fronteira. Cada classificador responde se a idade é igual ou superior a uma faixa. A probabilidade de cada classe sai da diferença entre dois classificadores consecutivos e obrigamos a que a probabilidade acumulada nunca aumente, uma condição que os 15 modelos independentes não garantem.

Cada fronteira usa todos os dados, o que ajuda as classes pequenas. O erro numa fronteira não arrasta as outras, ao contrário do que acontece no softmax de 16 vias.

Comparamos três formas de escolher a classe: a maior probabilidade, a mediana da distribuição e o valor esperado arredondado. Testamos também uma média com as probabilidades do XGBoost de referência.


In [3]:
E1 = carregar("e1_frank_hall")
if E1 is None:
    f1_arg, f1_med, f1_exp = [], [], []
    proba_fh = np.zeros((len(y), K))
    for fi, (tr, va) in enumerate(FOLDS):
        cum = np.ones((len(va), K + 1))
        for k in range(1, K):
            yk = (y[tr] >= k).astype(int)
            swk = compute_sample_weight("balanced", yk)
            m = XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss",
            )
            m.fit(Xv[tr], yk, sample_weight=swk)
            cum[:, k] = m.predict_proba(Xv[va])[:, 1]
        cum[:, 0], cum[:, K] = 1.0, 0.0
        cum = np.minimum.accumulate(cum, axis=1)  # monotonicidade
        p = np.clip(cum[:, :K] - cum[:, 1:], 0, None)
        p = p / p.sum(axis=1, keepdims=True)
        proba_fh[va] = p
        pred_arg = p.argmax(axis=1)
        pred_med = (cum[:, 1:K] >= 0.5).sum(axis=1)
        pred_exp = np.clip(np.round((p * CLASSES[None, :]).sum(axis=1)), 0, K - 1).astype(int)
        f1_arg.append(f1_score(y[va], pred_arg, average="macro"))
        f1_med.append(f1_score(y[va], pred_med, average="macro"))
        f1_exp.append(f1_score(y[va], pred_exp, average="macro"))
        print(f"  fold {fi}: argmax {f1_arg[-1]:.4f} | mediana {f1_med[-1]:.4f} | esperado {f1_exp[-1]:.4f}")
    np.savez(os.path.join(DATA_DIR, f"nb08_e1_proba{SUF}.npz"), proba=proba_fh)
    E1 = {"argmax": f1_arg, "mediana": f1_med, "esperado": f1_exp}
    gravar("e1_frank_hall", E1)

proba_fh = np.load(os.path.join(DATA_DIR, f"nb08_e1_proba{SUF}.npz"))["proba"]
registar("Ordinal cumulativo", "Frank & Hall (argmax)", E1["argmax"], base_f1s)
registar("Ordinal cumulativo", "Frank & Hall (mediana)", E1["mediana"], base_f1s)
registar("Ordinal cumulativo", "Frank & Hall (esperado)", E1["esperado"], base_f1s)
for w in [0.5, 0.3]:
    blend = w * proba_fh + (1 - w) * base_proba
    f1s = [f1_score(y[va], blend[va].argmax(axis=1), average="macro") for _, va in FOLDS]
    registar("Ordinal cumulativo", f"blend {w:.1f}*F&H + {1 - w:.1f}*XGB", f1s, base_f1s)
print(tabela(RESULTADOS[-5:]).to_string(index=False))

# Diagnóstico: o ganho é largo ou vem de poucas classes?
f1_cls = pd.DataFrame({
    "classe": CLASSES,
    "base": f1_score(y, base_pred, average=None),
    "F&H": f1_score(y, proba_fh.argmax(axis=1), average=None),
})
f1_cls["delta"] = f1_cls["F&H"] - f1_cls["base"]
print(f1_cls.round(3).to_string(index=False))

exp                variante     f1    std  delta  folds_ganhos nota
 E1  Frank & Hall (mediana) 0.4979 0.0131 0.0243             5     
 E1 blend 0.5*F&H + 0.5*XGB 0.4973 0.0088 0.0237             5     
 E1   Frank & Hall (argmax) 0.4935 0.0128 0.0199             5     
 E1 Frank & Hall (esperado) 0.4914 0.0089 0.0178             5     
 E1 blend 0.3*F&H + 0.7*XGB 0.4896 0.0058 0.0160             5     
 classe  base   F&H  delta
      0 0.862 0.864  0.001
      1 0.317 0.452  0.136
      2 0.603 0.595 -0.008
      3 0.396 0.425  0.028
      4 0.293 0.326  0.033
      5 0.363 0.375  0.012
      6 0.374 0.389  0.014
      7 0.395 0.435  0.040
      8 0.654 0.627 -0.027
      9 0.671 0.660 -0.011
     10 0.644 0.680  0.035
     11 0.527 0.516 -0.011
     12 0.377 0.361 -0.016
     13 0.289 0.323  0.034
     14 0.342 0.378  0.036
     15 0.477 0.493  0.016


## Refinamento pairwise das fronteiras

A confusão concentra-se em cerca de 13 pares de faixas vizinhas. Para cada par treinamos um classificador binário só com os casos dessas duas classes, o que transforma um problema de 16 classes num problema de duas. O classificador pairwise substitui a previsão base apenas quando o modelo principal hesita entre esse par.

Testamos duas regras. A primeira aplica o pairwise sempre que as duas classes mais prováveis são o par, o que cobre cerca de 80% das linhas. A segunda aplica-o apenas quando, além disso, a probabilidade máxima é inferior a 0.6, ou seja, quando há hesitação real.


In [4]:
E2 = carregar("e2_pairwise")
if E2 is None:
    f1_todos, f1_hesita = [], []
    cov_todos, cov_hesita, ac_hesita = [], [], []
    for fi, (tr, va) in enumerate(FOLDS):
        pred_a = base_pred[va].copy()
        pred_b = base_pred[va].copy()
        top2 = np.sort(np.argsort(base_proba[va], axis=1)[:, -2:], axis=1)
        maxp = base_proba[va].max(axis=1)
        cov_a = cov_b = ac_b = 0
        for a, b in PARES:
            mask_tr = np.isin(y[tr], [a, b])
            yb = (y[tr][mask_tr] == b).astype(int)
            m = XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss",
            )
            m.fit(Xv[tr][mask_tr], yb, sample_weight=compute_sample_weight("balanced", yb))
            sel = (top2[:, 0] == a) & (top2[:, 1] == b)
            if sel.sum() == 0:
                continue
            pb = m.predict_proba(Xv[va][sel])[:, 1]
            novo = np.where(pb >= 0.5, b, a)
            pred_a[sel] = novo
            cov_a += int(sel.sum())
            # variante B: só quando o multiclasse hesita (p_max < 0.6)
            sel_b = sel & (maxp < 0.6)
            if sel_b.sum():
                novo_b = np.where(pb[sel_b[sel]] >= 0.5, b, a)
                pred_b[sel_b] = novo_b
                cov_b += int(sel_b.sum())
                ac_b += int((novo_b == y[va][sel_b]).sum())
        f1_todos.append(f1_score(y[va], pred_a, average="macro"))
        f1_hesita.append(f1_score(y[va], pred_b, average="macro"))
        cov_todos.append(cov_a)
        cov_hesita.append(cov_b)
        ac_hesita.append(ac_b / max(cov_b, 1))
        print(f"  fold {fi}: todos {f1_todos[-1]:.4f} ({cov_a}) | hesita {f1_hesita[-1]:.4f} ({cov_b}, acc {ac_hesita[-1]:.3f})")
    E2 = {"todos": f1_todos, "hesita": f1_hesita,
          "cov_todos": cov_todos, "cov_hesita": cov_hesita, "ac_hesita": ac_hesita}
    gravar("e2_pairwise", E2)

registar("Pairwise", "pairwise (top-2 = par)", E2["todos"], base_f1s,
         nota=f"override em {np.mean(E2['cov_todos']):.0f} linhas/fold")
registar("Pairwise", "pairwise (top-2 = par e p_max<0.6)", E2["hesita"], base_f1s,
         nota=f"override em {np.mean(E2['cov_hesita']):.0f} linhas/fold, acc {np.mean(E2['ac_hesita']):.3f}")
print(tabela(RESULTADOS[-2:]).to_string(index=False))

exp                           variante     f1    std  delta  folds_ganhos                                    nota
 E2             pairwise (top-2 = par) 0.4829 0.0085 0.0093             5            override em 1775 linhas/fold
 E2 pairwise (top-2 = par e p_max<0.6) 0.4827 0.0052 0.0091             5 override em 1000 linhas/fold, acc 0.412


## Correção pela matriz de confusão

As escalas por classe dos notebooks anteriores corrigem apenas a diagonal, multiplicando a probabilidade de cada classe por um fator. Os erros são assimétricos. Por exemplo, a classe 4 é prevista como 5 em 158 casos e a classe 5 é prevista como 4 em 81. Uma matriz completa de 16 por 16, que estima a classe verdadeira para cada classe prevista, pode corrigir o que a diagonal não vê.

A matriz é estimada fora da fold que está a ser avaliada, para não haver fuga de informação. Testamos dois usos: substituir a classe prevista pela mais provável da matriz e corrigir as probabilidades antes de escolher a classe.


In [5]:
E3 = carregar("e3_confusao")
if E3 is None:
    f1_hard, f1_soft = [], []
    for fi, (tr, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        counts = np.zeros((K, K))
        np.add.at(counts, (y[outros], base_pred[outros]), 1)
        counts += 1.0
        Q = counts / counts.sum(axis=0, keepdims=True)  # Q[i,j]=P(true=i|pred=j)
        pred_h = Q.argmax(axis=0)[base_pred[va]]
        proba_s = base_proba[va] @ Q.T
        f1_hard.append(f1_score(y[va], pred_h, average="macro"))
        f1_soft.append(f1_score(y[va], proba_s.argmax(axis=1), average="macro"))
        print(f"  fold {fi}: dura {f1_hard[-1]:.4f} | suave {f1_soft[-1]:.4f}")
    E3 = {"dura": f1_hard, "suave": f1_soft}
    gravar("e3_confusao", E3)

registar("Matriz de confusão", "correcao confusao 16x16 (dura)", E3["dura"], base_f1s)
registar("Matriz de confusão", "correcao confusao 16x16 (suave)", E3["suave"], base_f1s)
print(tabela(RESULTADOS[-2:]).to_string(index=False))

exp                        variante     f1    std   delta  folds_ganhos nota
 E3  correcao confusao 16x16 (dura) 0.4569 0.0135 -0.0167             0     
 E3 correcao confusao 16x16 (suave) 0.4512 0.0095 -0.0224             0     


## Hierarquia grosso-fino

Dividimos as 16 faixas em quatro grupos com significado de ciclo de vida: 15-24, 25-54, 55-69 e 70+. Um primeiro modelo prevê o grupo e um segundo modelo, treinado apenas com os casos desse grupo, prevê a faixa dentro dele.

A probabilidade final é o produto da probabilidade do grupo pela probabilidade da faixa dentro do grupo. O routing do questionário separa os grupos com facilidade, sobretudo o corte dos 55 anos. O segundo nível fica com um problema mais pequeno, entre três e seis faixas vizinhas.


In [6]:
E4 = carregar("e4_hierarquia")
if E4 is None:
    GRUPOS = {0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8], 2: [9, 10, 11], 3: [12, 13, 14, 15]}
    g = np.full(len(y), -1, dtype=int)
    for gi, classes in GRUPOS.items():
        g[np.isin(y, classes)] = gi
    f1s = []
    proba_e4 = np.zeros((len(y), K))
    for fi, (tr, va) in enumerate(FOLDS):
        m1 = XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss",
        )
        m1.fit(Xv[tr], g[tr], sample_weight=compute_sample_weight("balanced", g[tr]))
        pg = m1.predict_proba(Xv[va])  # (n, 4)
        p_full = np.zeros((len(va), K))
        for gi, classes in GRUPOS.items():
            mask_tr = np.isin(y[tr], classes)
            # XGB exige labels 0..G-1: remapear dentro do grupo
            remap = {c: i for i, c in enumerate(classes)}
            y_enc = np.array([remap[v] for v in y[tr][mask_tr]])
            m2 = XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss",
            )
            m2.fit(Xv[tr][mask_tr], y_enc,
                   sample_weight=compute_sample_weight("balanced", y_enc))
            p_full[:, classes] = pg[:, [gi]] * m2.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], p_full.argmax(axis=1), average="macro"))
        proba_e4[va] = p_full
        print(f"  fold {fi}: {f1s[-1]:.4f}")
    np.savez(os.path.join(DATA_DIR, f"nb08_e4_proba{SUF}.npz"), proba=proba_e4)
    E4 = {"f1": f1s}
    gravar("e4_hierarquia", E4)

registar("Hierarquia", "coarse->fine (4 grupos)", E4["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))

exp                variante     f1    std  delta  folds_ganhos nota
 E4 coarse->fine (4 grupos) 0.4842 0.0067 0.0106             5     


## Alvos suavizados pela distância entre faixas

O softmax trata as 16 classes como símbolos independentes. Errar por uma faixa ou por oito custa o mesmo. Nesta experiência o alvo é uma distribuição centrada na classe verdadeira, com probabilidade que decresce com a distância. O gradiente passa a ser a diferença entre a distribuição prevista e essa distribuição alvo.

O modelo deixa de estar confiantemente errado na fronteira e reparte massa pelas classes vizinhas. O parâmetro `tau` controla a largura da distribuição. Um `tau` pequeno aproxima-se do softmax normal e serve de controlo.

O objetivo é implementado com `xgb.train`, porque o wrapper de sklearn não expõe bem o hessiano multiclasse.


In [7]:
def _softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def _obj_distancia(tau):
    grid = np.arange(K, dtype=float)

    def obj(preds, dtrain):
        yt = dtrain.get_label().astype(int)
        p = _softmax(preds)
        q = np.exp(-np.abs(grid[None, :] - yt[:, None]) / tau)
        q = q / q.sum(axis=1, keepdims=True)
        return p - q, np.clip(p * (1 - p), 1e-6, None)

    return obj


E5 = carregar("e5_softlabels")
if E5 is None:
    E5 = {}
    for tau in [0.5, 1.0]:
        f1s = []
        for fi, (tr, va) in enumerate(FOLDS):
            dtr = xgb.DMatrix(Xv[tr], label=y[tr], weight=SW_BAL[tr])
            params = {
                "num_class": K, "max_depth": 6, "eta": 0.05, "subsample": 0.8,
                "colsample_bytree": 0.8, "tree_method": "hist",
                "nthread": -1, "seed": RANDOM_STATE,
            }
            bst = xgb.train(params, dtr, num_boost_round=500, obj=_obj_distancia(tau))
            marg = bst.predict(xgb.DMatrix(Xv[va]), output_margin=True)
            f1s.append(f1_score(y[va], _softmax(marg).argmax(axis=1), average="macro"))
        E5[f"tau={tau}"] = f1s
        print(f"  tau={tau}: {np.mean(f1s):.4f}")
    gravar("e5_softlabels", E5)

for nome, f1s in E5.items():
    registar("Alvos suaves", f"soft labels {nome}", f1s, base_f1s)
print(tabela(RESULTADOS[-len(E5):]).to_string(index=False))

exp            variante     f1    std  delta  folds_ganhos nota
 E5 soft labels tau=0.5 0.4816 0.0074 0.0080             4     
 E5 soft labels tau=1.0 0.4785 0.0154 0.0049             4     


## Peso das classes

O projeto usa `class_weight="balanced"`, que dá mais peso às classes raras. Com 16 classes, este peso pode estar a exagerar e a tornar as probabilidades demasiado confiantes. Testamos três valores para o expoente dos pesos: 0, que corresponde a não usar pesos, 0.5 e 1.

O valor 1 deve reproduzir exatamente a referência. Esta repetição serve de controlo: se não reproduzir, há um erro na experiência.


In [8]:
E6 = carregar("e6_pesos")
if E6 is None:
    E6 = {}
    for alpha in [0.0, 0.5, 1.0]:
        if alpha == 0:
            sw = None
        else:
            w = (1.0 / FREQ) ** alpha
            w = w / (w * FREQ).sum()  # média amostral = 1 (alpha=1 == balanced)
            sw = w[y]
        pred = np.empty(len(y), dtype=int)
        proba = np.zeros((len(y), K))
        f1s = []
        for tr, va in FOLDS:
            m = XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss",
            )
            m.fit(Xv[tr], y[tr], sample_weight=None if sw is None else sw[tr])
            pred[va] = m.predict(Xv[va])
            proba[va] = m.predict_proba(Xv[va])
            f1s.append(f1_score(y[va], pred[va], average="macro"))
        E6[f"alpha={alpha}"] = {"f1": f1s, "oof": pred, "proba": proba}
        print(f"  alpha={alpha}: {np.mean(f1s):.4f}")
    np.savez(
        os.path.join(DATA_DIR, f"nb08_e6_pesos{SUF}.npz"),
        **{f"oof_{k}": v["oof"] for k, v in E6.items()},
        **{f"proba_{k}": v["proba"] for k, v in E6.items()},
    )
    gravar("e6_pesos", {k: v["f1"] for k, v in E6.items()})

# compatibilidade com o checkpoint achatado {nome: [f1 por fold]}
E6 = {k: (v if isinstance(v, dict) else {"f1": v}) for k, v in E6.items()}

for nome, v in E6.items():
    registar("Peso das classes", f"sample_weight {nome}", v["f1"], base_f1s)
print(tabela(RESULTADOS[-len(E6):]).to_string(index=False))

exp                variante     f1    std  delta  folds_ganhos nota
 E6 sample_weight alpha=0.5 0.4750 0.0077 0.0014             2     
 E6 sample_weight alpha=0.0 0.4739 0.0076 0.0003             3     
 E6 sample_weight alpha=1.0 0.4736 0.0062 0.0000             0     


## Calibração das probabilidades

Quase metade das previsões com probabilidade igual ou superior a 0.6 está errada. A calibração corrige a escala das probabilidades sem mudar a classe escolhida, por isso só interessa se melhorar decisões a jusante.

Testamos duas calibrações, a temperature scaling, com um parâmetro, e a vector scaling, com um parâmetro por classe. Ambas são ajustadas fora da fold avaliada. Medimos o erro logarítmico, o Brier score e o erro de calibração esperado em 10 bins.

Depois testamos duas decisões: as escalas por classe e o limiar do stacking entre classificador e regressor. Ao contrário dos notebooks anteriores, os parâmetros são ajustados fora da fold, para não repetir o otimismo da avaliação anterior.


In [9]:
E7 = carregar("e7_calibracao")
if E7 is None:
    logp = np.log(np.clip(base_proba, 1e-12, None))

    def _ece(y_true, proba, bins=10):
        conf = proba.max(axis=1)
        acc = (proba.argmax(axis=1) == y_true).astype(float)
        edges = np.linspace(0, 1, bins + 1)
        e = 0.0
        for i in range(bins):
            m = (conf > edges[i]) & (conf <= edges[i + 1])
            if m.sum():
                e += m.mean() * abs(acc[m].mean() - conf[m].mean())
        return float(e)

    def _nll(y_true, proba):
        return float(-np.mean(np.log(np.clip(proba[np.arange(len(y_true)), y_true], 1e-12, None))))

    def _brier(y_true, proba):
        oh = np.zeros_like(proba)
        oh[np.arange(len(y_true)), y_true] = 1
        return float(np.mean(((proba - oh) ** 2).sum(axis=1)))

    proba_T = np.zeros_like(base_proba)
    Ts = []
    for fi, (tr, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        lp, yy = logp[outros], y[outros]
        f = lambda t: _nll(yy, _softmax(lp / t))
        r = minimize_scalar(f, bounds=(0.2, 5.0), method="bounded")
        Ts.append(float(r.x))
        proba_T[va] = _softmax(logp[va] / r.x)

    def _vector_scale(lp, yy, l2=1e-3):
        n, k = lp.shape
        x0 = np.concatenate([np.ones(k), np.zeros(k)])

        def fun(theta):
            w, b = theta[:k], theta[k:]
            p = _softmax(lp * w + b)
            nll = -np.mean(np.log(np.clip(p[np.arange(n), yy], 1e-12, None)))
            return nll + l2 * np.sum(w**2)

        r = minimize(fun, x0, method="L-BFGS-B", options={"maxiter": 300})
        return r.x[:k], r.x[k:]

    proba_V = np.zeros_like(base_proba)
    for fi, (tr, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        w, b = _vector_scale(logp[outros], y[outros])
        proba_V[va] = _softmax(logp[va] * w + b)

    metricas = []
    for nome, p in [("crua", base_proba), ("temperature", proba_T), ("vector", proba_V)]:
        metricas.append({
            "calibracao": nome, "NLL": _nll(y, p), "Brier": _brier(y, p),
            "ECE": _ece(y, p), "F1_argmax": f1_score(y, p.argmax(axis=1), average="macro"),
        })

    def _tune_escalas(proba, yt, rondas=2):
        esc = np.ones(proba.shape[1])
        base = f1_score(yt, proba.argmax(axis=1), average="macro")
        for _ in range(rondas):
            for k in range(proba.shape[1]):
                for gm in np.arange(0.5, 2.05, 0.25):
                    t = esc.copy()
                    t[k] = gm
                    aj = proba * t
                    s = f1_score(yt, aj.argmax(axis=1), average="macro")
                    if s > base:
                        base, esc = s, t
        return esc

    f1_esc_raw, f1_esc_cal = [], []
    for fi, (tr, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        e1 = _tune_escalas(base_proba[outros], y[outros])
        e2 = _tune_escalas(proba_T[outros], y[outros])
        f1_esc_raw.append(f1_score(y[va], (base_proba[va] * e1).argmax(axis=1), average="macro"))
        f1_esc_cal.append(f1_score(y[va], (proba_T[va] * e2).argmax(axis=1), average="macro"))

    reg = np.empty(len(y))
    for tr, va in FOLDS:
        mr = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
        mr.fit(X.iloc[tr], y[tr])
        reg[va] = np.clip(np.round(mr.predict(X.iloc[va])), 0, K - 1).astype(int)

    def _stack_crossfit(proba, nome):
        f1s, ts = [], []
        for fi, (tr, va) in enumerate(FOLDS):
            outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
            best_t, best_f = 0.5, -1.0
            for t in np.arange(0.05, 0.65, 0.05):
                p = np.where(proba[outros].max(axis=1) >= t,
                             proba[outros].argmax(axis=1), reg[outros])
                f = f1_score(y[outros], p, average="macro")
                if f > best_f:
                    best_f, best_t = float(f), float(t)
            ts.append(best_t)
            pv = np.where(proba[va].max(axis=1) >= best_t, proba[va].argmax(axis=1), reg[va])
            f1s.append(f1_score(y[va], pv, average="macro"))
        return f1s, ts

    f1_stack_raw, t_raw = _stack_crossfit(base_proba, "crua")
    f1_stack_cal, t_cal = _stack_crossfit(proba_T, "calibrada")

    E7 = {
        "metricas": metricas, "T": Ts,
        "escalas_cru": f1_esc_raw, "escalas_cal": f1_esc_cal,
        "stack_cru": f1_stack_raw, "stack_cal": f1_stack_cal, "t": t_raw,
        "regressor_f1": float(f1_score(y, reg, average="macro")),
    }
    gravar("e7_calibracao", E7)

print("Calibracao:")
print(pd.DataFrame(E7["metricas"]).round(4).to_string(index=False))
print("T por fold:", np.round(E7["T"], 3).tolist())
registar("Calibração", "escalas cross-fitted (cruas)", E7["escalas_cru"], base_f1s)
registar("Calibração", "escalas cross-fitted (calibradas)", E7["escalas_cal"], base_f1s)
registar("Calibração", "stack t* cross-fitted (cru)", E7["stack_cru"], base_f1s,
         nota=f"regressor sozinho {E7['regressor_f1']:.4f}")
registar("Calibração", "stack t* cross-fitted (calibrado)", E7["stack_cal"], base_f1s)
print(tabela(RESULTADOS[-4:]).to_string(index=False))

Calibracao:
 calibracao    NLL  Brier    ECE  F1_argmax
       crua 1.2391 0.6416 0.1054     0.4740
temperature 1.2183 0.6277 0.0484     0.4740
     vector 1.2147 0.6253 0.0470     0.4706
T por fold: [1.247, 1.244, 1.242, 1.225, 1.245]
exp                          variante     f1    std  delta  folds_ganhos                     nota
 E7       stack t* cross-fitted (cru) 0.4791 0.0056 0.0055             4 regressor sozinho 0.4178
 E7 escalas cross-fitted (calibradas) 0.4790 0.0073 0.0054             4                         
 E7 stack t* cross-fitted (calibrado) 0.4778 0.0050 0.0042             4                         
 E7      escalas cross-fitted (cruas) 0.4775 0.0068 0.0039             5                         


## Combinação dos métodos

A decomposição cumulativa, a hierarquia e o refinamento pairwise atacam a fronteira por mecanismos diferentes. Se os erros não forem os mesmos, a combinação pode somar.

Testamos médias de probabilidades entre a decomposição cumulativa e a hierarquia, com e sem o XGBoost base. Testamos também o refinamento pairwise aplicado à decomposição cumulativa, apenas quando ela hesita. A fasquia passa a ser o melhor resultado isolado, 0.4979, e não a referência inicial.


In [10]:
E8 = carregar("e8_combinacoes")
if E8 is None:
    p_e4 = np.load(os.path.join(DATA_DIR, f"nb08_e4_proba{SUF}.npz"))["proba"]
    combos = {
        "F&H + E4 (0.5/0.5)": 0.5 * proba_fh + 0.5 * p_e4,
        "F&H + E4 (0.7/0.3)": 0.7 * proba_fh + 0.3 * p_e4,
        "F&H + E4 + XGB (1/3)": (proba_fh + p_e4 + base_proba) / 3.0,
    }
    E8 = {}
    for nome, p in combos.items():
        f1s = [f1_score(y[va], p[va].argmax(axis=1), average="macro") for _, va in FOLDS]
        E8[nome] = f1s
        print(f"  {nome}: {np.mean(f1s):.4f}")

    f1s = []
    for fi, (tr, va) in enumerate(FOLDS):
        pred = proba_fh[va].argmax(axis=1).copy()
        top2 = np.sort(np.argsort(proba_fh[va], axis=1)[:, -2:], axis=1)
        maxp = proba_fh[va].max(axis=1)
        for a, b in PARES:
            mask_tr = np.isin(y[tr], [a, b])
            yb = (y[tr][mask_tr] == b).astype(int)
            m = XGBClassifier(
                n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss",
            )
            m.fit(Xv[tr][mask_tr], yb, sample_weight=compute_sample_weight("balanced", yb))
            sel = (top2[:, 0] == a) & (top2[:, 1] == b) & (maxp < 0.6)
            if sel.sum():
                pred[sel] = np.where(m.predict_proba(Xv[va][sel])[:, 1] >= 0.5, b, a)
        f1s.append(f1_score(y[va], pred, average="macro"))
        print(f"  fold {fi} F&H+pairwise: {f1s[-1]:.4f}")
    E8["F&H + pairwise (hesita)"] = f1s
    gravar("e8_combinacoes", E8)

rows8 = [registar("Combinações", nome, f1s, base_f1s) for nome, f1s in E8.items()]
f1_fh_ref = float(np.mean([
    f1_score(y[va], proba_fh[va].argmax(axis=1), average="macro") for _, va in FOLDS
]))
f1_fh_med = 0.4979  # decodificador de mediana do E1 (fasquia a bater)
tab8 = pd.DataFrame(rows8).sort_values("f1", ascending=False)
tab8["delta_F&H"] = (tab8["f1"] - f1_fh_med).round(4)
tab8 = tab8.round(4)
print(tab8.to_string(index=False))
print(f"F&H argmax = {f1_fh_ref:.4f} | F&H mediana = {f1_fh_med:.4f} (fasquia)")

exp                variante     f1    std  delta  folds_ganhos nota  delta_F&H
 E8      F&H + E4 (0.5/0.5) 0.4982 0.0093 0.0246             5          0.0003
 E8 F&H + pairwise (hesita) 0.4981 0.0100 0.0245             5          0.0002
 E8      F&H + E4 (0.7/0.3) 0.4970 0.0091 0.0234             5         -0.0009
 E8    F&H + E4 + XGB (1/3) 0.4960 0.0087 0.0224             5         -0.0019
F&H argmax = 0.4935 | F&H mediana = 0.4979 (fasquia)


## Experiências adicionais

As secções seguintes alargam o leque de testes. Umas atacam o treino, como o peso das linhas difíceis e os scores pairwise usados como features. Outras testam modelos alternativos, como LightGBM, CatBoost e uma rede neuronal. As últimas verificam se a média de seeds, os limiares por classe e o meta-stacking acrescentam algo.

Cada bloco tem a mesma estrutura: a pergunta, o método e a leitura do resultado.


## Ajuste de pesos nas linhas que o modelo erra

Uma forma de concentrar o treino nas fronteiras é dar mais peso às linhas que o modelo de referência errou. O erro é calculado nas previsões out-of-fold, por isso a linha nunca foi usada para se prever a si própria.

Testamos dois fatores de reforço, 1 e 2, multiplicados pelo peso balanceado original. O risco é reforçar ruído e não sinal, sobretudo nas classes pequenas.


In [ ]:
E9 = carregar("e9_pesos_fronteira")
if E9 is None:
    E9 = {}
    # Upweight das linhas que a base erra (quase todas off-by-one de fronteira).
    # `falhou` vem da OOF: cada linha foi prevista sem treinar nela -> sem leakage.
    falhou = (base_pred != y).astype(float)
    for alpha in [1.0, 2.0]:
        sw = SW_BAL * (1.0 + alpha * falhou)
        f1s = []
        for tr, va in FOLDS:
            m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                              random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
            m.fit(Xv[tr], y[tr], sample_weight=sw[tr])
            f1s.append(f1_score(y[va], m.predict(Xv[va]), average="macro"))
        E9[f"miss x{alpha}"] = f1s
        print(f"  alpha={alpha}: {np.mean(f1s):.4f}")
    gravar("e9_pesos_fronteira", E9)
for k, v in E9.items():
    registar("Pesos fronteira", f"pesos fronteira ({k})", v, base_f1s)
print(tabela(RESULTADOS[-len(E9):]).to_string(index=False))


## Scores pairwise como features

Em vez de usar o classificador pairwise para substituir a previsão, usamos a probabilidade de cada par como uma feature nova. Treinamos um modelo final com as 339 features e mais sete scores pairwise, um por cada par de fronteiras com mais confusão.

A probabilidade de cada par é calculada out-of-fold, para não haver fuga. O modelo final pode combinar estes scores com o resto do perfil.


In [ ]:
QUENTES = [(3, 4), (4, 5), (5, 6), (6, 7), (12, 13), (13, 14), (14, 15)]
E10 = carregar("e10_pairwise_soft")
if E10 is None:
    # Probabilidades out-of-fold dos pares de fronteira usadas como features.
    S_oof = np.zeros((len(y), len(QUENTES)))
    for j, (a, b) in enumerate(QUENTES):
        for tr, va in FOLDS:
            sel_tr = np.isin(y[tr], [a, b])
            yb = (y[tr][sel_tr] == b).astype(int)
            m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                              random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
            m.fit(Xv[tr][sel_tr], yb, sample_weight=compute_sample_weight("balanced", yb))
            S_oof[va, j] = m.predict_proba(Xv[va])[:, 1]
        print(f"  par {(a, b)}: ok")
    Xs = np.hstack([Xv, S_oof])
    f1s = []
    for tr, va in FOLDS:
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
        m.fit(Xs[tr], y[tr], sample_weight=SW_BAL[tr])
        f1s.append(f1_score(y[va], m.predict(Xs[va]), average="macro"))
    E10 = {"f1": f1s}
    np.savez(os.path.join(DATA_DIR, f"nb08_e10_pairwise_soft{SUF}.npz"), S_oof=S_oof)
    # Scores no teste (fits no treino completo) para reutilizar numa futura submissao.
    te = pd.read_csv(os.path.join(DATA_DIR, "test_nb02.csv"))
    Xte = te[X.columns].to_numpy()
    S_te = np.zeros((len(te), len(QUENTES)))
    for j, (a, b) in enumerate(QUENTES):
        sel = np.isin(y, [a, b])
        yb = (y[sel] == b).astype(int)
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
        m.fit(Xv[sel], yb, sample_weight=compute_sample_weight("balanced", yb))
        S_te[:, j] = m.predict_proba(Xte)[:, 1]
    np.savez(os.path.join(DATA_DIR, f"nb08_e10_pairwise_soft_te{SUF}.npz"), S_te=S_te)
    gravar("e10_pairwise_soft", E10)
registar("Pairwise suave", "pairwise suave como features", E10["f1"], base_f1s,
         nota=f"{len(QUENTES)} pares quentes")
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## Indicador de recusador

O Notebook 01 mostrou que algumas pessoas respondem "Sem resposta" a muitas perguntas. Esta experiência conta essas respostas no texto original e acrescenta ao modelo a contagem e um indicador para quem tem mais de 20 não-respostas.

O alinhamento é feito por `PID`, porque a ordem dos CSV não é garantida. A contagem é uma medida de qualidade da entrevista e pode ajudar a explicar perfis ambíguos.


In [ ]:
E11 = carregar("e11_recusadores")
if E11 is None:
    # N_SEM_RESPOSTA nao esta na base 339 (item §7.3 aberto): conta + flag >20.
    # Alinhamento por PID (ordem dos CSVs nao e garantida).
    t01 = pd.read_csv(os.path.join(DATA_DIR, "train_nb01.csv"), keep_default_na=False, na_values=[""])
    v01 = pd.read_csv(os.path.join(DATA_DIR, "test_nb01.csv"), keep_default_na=False, na_values=[""])
    def conta_refusa(d):
        txt = d.select_dtypes(include="str")
        return ((txt == "Sem resposta") | (txt == "NR")).sum(axis=1)
    r_tr = conta_refusa(t01)
    r_tr.index = t01["PID"].to_numpy()
    r_te = conta_refusa(v01)
    r_te.index = v01["PID"].to_numpy()
    NREF_tr = r_tr.loc[train["PID"].to_numpy()].to_numpy().astype(float)
    te = pd.read_csv(os.path.join(DATA_DIR, "test_nb02.csv"))
    NREF_te = r_te.loc[te["PID"].to_numpy()].to_numpy().astype(float)
    print(f"recusadores treino>20: {(NREF_tr > 20).sum()} | teste>20: {(NREF_te > 20).sum()}")
    Xs_tr = np.hstack([Xv, NREF_tr[:, None], (NREF_tr > 20).astype(float)[:, None]])
    f1s = []
    for tr, va in FOLDS:
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
        m.fit(Xs_tr[tr], y[tr], sample_weight=SW_BAL[tr])
        f1s.append(f1_score(y[va], m.predict(Xs_tr[va]), average="macro"))
    E11 = {"f1": f1s}
    np.savez(os.path.join(DATA_DIR, f"nb08_e11_recusadores{SUF}.npz"), NREF_tr=NREF_tr, NREF_te=NREF_te)
    gravar("e11_recusadores", E11)
registar("Recusadores", "flag recusador + N_SEM_RESPOSTA", E11["f1"], base_f1s,
         nota="161 recusadores (>20) no treino")
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## Meta-stacking

As experiências anteriores produzem várias probabilidades out-of-fold: XGBoost, HGB, decomposição cumulativa e hierarquia. O meta-stacking treina um modelo simples, regressão logística, sobre essas probabilidades.

O meta-modelo treina em quatro folds e prevê a quinta, por isso a avaliação é honesta. Comparamos duas regularizações, C igual a 1 e C igual a 0.1.


In [ ]:
E12 = carregar("e12_metastack")
if E12 is None:
    # Meta-modelo treinado e avaliado fora da fold, com XGBoost, HGB, decomposição cumulativa e hierarquia.
    # Sem refits das bases: usa as OOFs guardadas (todas nas mesmas folds seed 42)
    # -> custo de segundos. O meta treina em 4 folds e preve o 5.o (honesto).
    from sklearn.linear_model import LogisticRegression
    zx = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))["proba"]
    zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
    zf = np.load(os.path.join(DATA_DIR, "nb08_e1_proba.npz"))["proba"]
    ze = np.load(os.path.join(DATA_DIR, "nb08_e4_proba.npz"))["proba"]
    M = np.hstack([zx, zh, zf, ze])
    print("meta-features:", M.shape)
    E12 = {}
    for nome, C in [("logreg C=1.0", 1.0), ("logreg C=0.1", 0.1)]:
        f1s = []
        for tr, va in FOLDS:
            meta = LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
            meta.fit(M[tr], y[tr])
            f1s.append(f1_score(y[va], meta.predict(M[va]), average="macro"))
        E12[nome] = f1s
        print(f"  {nome}: {np.mean(f1s):.4f}")
    gravar("e12_metastack", E12)
for k, v in E12.items():
    registar("Meta-stacking", f"metastack ({k})", v, base_f1s)
print(tabela(RESULTADOS[-len(E12):]).to_string(index=False))


## Rede neuronal

Testamos uma rede neuronal simples para fechar a pergunta sobre modelos não lineares. Usamos um MLP com uma camada de 256 unidades, padronização das features dentro de cada fold e paragem antecipada.

É uma configuração única, não uma pesquisa. O objetivo é saber se uma rede neuronal se aproxima dos modelos de árvores neste problema.


In [ ]:
E13 = carregar("e13_mlp")
if E13 is None:
    # Trial neuronal limitado (1 config): MLP sklearn em CPU, com scaler in-fold
    # e early stopping. Fecha a pergunta "e redes neuronais?" com numero.
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler
    f1s, iters = [], []
    for tr, va in FOLDS:
        sc = StandardScaler()
        Ztr, Zva = sc.fit_transform(Xv[tr]), sc.transform(Xv[va])
        m = MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, max_iter=1000,
                          early_stopping=True, validation_fraction=0.15,
                          n_iter_no_change=20, random_state=RANDOM_STATE)
        m.fit(Ztr, y[tr])
        iters.append(int(m.n_iter_))
        f1s.append(f1_score(y[va], m.predict(Zva), average="macro"))
    E13 = {"f1": f1s, "n_iter": iters}
    gravar("e13_mlp", E13)
registar("MLP", "MLP (256) early-stopping + scaler", E13["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## Média de várias seeds

O XGBoost tem variância entre execuções. Esta experiência treina o mesmo modelo com três seeds, 42, 7 e 123, e faz a média das probabilidades.

A média reduz a variância sem mudar o algoritmo. Em modo rápido, a experiência mistura esquemas de folds diferentes e serve apenas de teste de fumo.


In [ ]:
E14 = carregar("e14_seedavg")
if E14 is None:
    # Seed-average do XGB: base (seed 42) + 2 seeds novas; media das probas OOF.
    # Em rapido mistura OOFs de esquemas diferentes (5-fold + 2-fold): so smoke.
    P = base_proba.copy()
    for seed in [7, 123]:
        for tr, va in FOLDS:
            m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                              random_state=seed, n_jobs=-1, eval_metric="mlogloss")
            m.fit(Xv[tr], y[tr], sample_weight=SW_BAL[tr])
            P[va] += m.predict_proba(Xv[va])
    P /= 3.0
    f1s = [f1_score(y[va], P[va].argmax(axis=1), average="macro") for _, va in FOLDS]
    E14 = {"f1": f1s}
    np.savez(os.path.join(DATA_DIR, f"nb08_e14_seedavg{SUF}.npz"), proba=P)
    gravar("e14_seedavg", E14)
registar("Seed average", "seed-average XGB (42, 7, 123)", E14["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## Limiares por classe

As escalas por classe ajustam a probabilidade de cada classe. Esta experiência faz o mesmo no espaço dos logaritmos, somando um valor a cada classe antes de escolher a mais provável. São 16 parâmetros, ajustados por subida coordenada.

O ajuste é feito fora da fold avaliada. Sem esta precaução, o resultado repetiria o otimismo do limiar do stacking usado nos notebooks anteriores.


In [ ]:
E15 = carregar("e15_thresholds")
if E15 is None:
    # Bias por classe (16 params) por coordinate ascent, sempre cross-fitted:
    # o bias do fold fi e sintonizado nos OUTROS folds. Sem este cuidado seria
    # o mesmo otimismo OOF do t* antigo (+1pp -> +0.0055).
    logp = np.log(np.clip(base_proba, 1e-12, None))
    def _f1b(b, idx):
        return f1_score(y[idx], (logp[idx] + b[None, :]).argmax(axis=1), average="macro")
    B = np.zeros((len(FOLDS), K))
    for fi, (tr, va) in enumerate(FOLDS):
        outros = np.concatenate([v for fj, (_, v) in enumerate(FOLDS) if fj != fi])
        b = np.zeros(K)
        base = _f1b(b, outros)
        for _ in range(3):
            for k in range(K):
                for d in (0.05, -0.05, 0.1, -0.1):
                    b2 = b.copy()
                    b2[k] += d
                    f = _f1b(b2, outros)
                    if f > base + 1e-6:
                        base, b = f, b2
        B[fi] = b
    pred = np.empty(len(y), dtype=int)
    for fi, (_, va) in enumerate(FOLDS):
        pred[va] = (logp[va] + B[fi][None, :]).argmax(axis=1)
    E15 = {"f1": [float(f1_score(y[va], pred[va], average="macro")) for _, va in FOLDS],
           "B": B.tolist()}
    gravar("e15_thresholds", E15)
registar("Limiares", "thresholds por classe (cross-fitted)", E15["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## LightGBM

O LightGBM foi instalado no ambiente em 27 de outubro. Usamos o mesmo protocolo e hiperparâmetros comparáveis ao XGBoost: 500 árvores, 31 folhas, learning rate 0.05, subsample e feature fraction 0.8, e pesos balanceados.

Esta experiência responde se o algoritmo se comporta melhor do que o XGBoost neste conjunto de dados.


In [ ]:
E16 = carregar("e16_lightgbm")
if E16 is None:
    # LightGBM 4.6.0 (instalado 27 out via conda): mesmo protocolo, hiperparams
    # emparelhados com o XGB (500 arvores, leaves 31, lr 0.05, subsample 0.8).
    import lightgbm as lgb
    oof = np.empty(len(y), dtype=int)
    pro = np.zeros((len(y), K))
    f1s = []
    for tr, va in FOLDS:
        m = lgb.LGBMClassifier(n_estimators=500, num_leaves=31, learning_rate=0.05,
                               feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                               lambda_l2=1.0, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
        m.fit(Xv[tr], y[tr])
        oof[va] = m.predict(Xv[va])
        pro[va] = m.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], oof[va], average="macro"))
    E16 = {"f1": f1s}
    np.savez(os.path.join(DATA_DIR, f"nb08_e16_lightgbm{SUF}.npz"), oof=oof, proba=pro)
    gravar("e16_lightgbm", E16)
registar("LightGBM", "LightGBM (500, leaves 31)", E16["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## CatBoost

O CatBoost também foi instalado em 27 de outubro. Usamos 500 iterações, profundidade 6, learning rate 0.05 e pesos de classe balanceados.

O CatBoost trata variáveis categóricas de forma nativa, mas aqui todas as features já são numéricas. Esta experiência mede sobretudo o efeito do algoritmo de boosting.


In [ ]:
E17 = carregar("e17_catboost")
if E17 is None:
    # CatBoost 1.2.10 (instalado 27 out via conda): mesmo protocolo.
    from catboost import CatBoostClassifier
    oof = np.empty(len(y), dtype=int)
    pro = np.zeros((len(y), K))
    f1s = []
    for tr, va in FOLDS:
        m = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                               auto_class_weights="Balanced", random_seed=RANDOM_STATE,
                               thread_count=-1, verbose=False, allow_writing_files=False)
        m.fit(Xv[tr], y[tr])
        oof[va] = np.asarray(m.predict(Xv[va]), dtype=int).ravel()
        pro[va] = m.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], oof[va], average="macro"))
    E17 = {"f1": f1s}
    np.savez(os.path.join(DATA_DIR, f"nb08_e17_catboost{SUF}.npz"), oof=oof, proba=pro)
    gravar("e17_catboost", E17)
registar("CatBoost", "CatBoost (500, depth 6)", E17["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


## Meta-stacking com cinco bases

Esta variante repete o meta-stacking com cinco famílias de probabilidades: XGBoost com média de seeds, HGB, decomposição cumulativa, hierarquia e LightGBM. O meta-modelo recebe 80 features e volta a ser uma regressão logística com duas regularizações.

O CatBoost fica de fora porque foi rejeitado na experiência anterior. O objetivo é ver se juntar mais famílias melhora a decisão do meta-modelo ou apenas acrescenta ruído.


In [ ]:
E12b = carregar("e12b_metastack5")
if E12b is None:
    # Cinco famílias de probabilidades, 80 features. O CatBoost fica de fora por ter sido rejeitado.
    from sklearn.linear_model import LogisticRegression
    zxs = np.load(os.path.join(DATA_DIR, f"nb08_e14_seedavg{SUF}.npz"))["proba"]
    zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
    zf = np.load(os.path.join(DATA_DIR, f"nb08_e1_proba{SUF}.npz"))["proba"]
    ze = np.load(os.path.join(DATA_DIR, f"nb08_e4_proba{SUF}.npz"))["proba"]
    zl = np.load(os.path.join(DATA_DIR, f"nb08_e16_lightgbm{SUF}.npz"))["proba"]
    M5 = np.hstack([zxs, zh, zf, ze, zl])
    print("meta-features:", M5.shape)
    E12b = {}
    for nome, C in [("logreg C=1.0", 1.0), ("logreg C=0.1", 0.1)]:
        f1s = []
        for tr, va in FOLDS:
            meta = LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
            meta.fit(M5[tr], y[tr])
            f1s.append(f1_score(y[va], meta.predict(M5[va]), average="macro"))
        E12b[nome] = f1s
        print(f"  {nome}: {np.mean(f1s):.4f}")
    gravar("e12b_metastack5", E12b)
for k, v in E12b.items():
    registar("Meta-stacking 5 bases", f"metastack5 ({k})", v, base_f1s)
print(tabela(RESULTADOS[-len(E12b):]).to_string(index=False))


## Tabela final

A tabela reúne todas as experiências. A coluna `f1` tem a média das cinco folds e a coluna `std` o desvio-padrão. A coluna `delta` compara com o XGBoost de referência e `folds_ganhos` conta em quantas folds a variante ganhou.

A regra de adoção é ganhar pelo menos 0.005 e vencer em quatro folds. Para substituir a decomposição cumulativa, a variante tem de bater também 0.4979. A tabela é gravada em `data/nb08_resultados.csv`.


In [ ]:
E12c = carregar("e12c_metastack_full")
if E12c is None:
    # Variante do artigo TDS: o meta recebe TAMBEM as 339 originais (standardizadas
    # in-fold) + as 64 probas OOF. E a unica diferenca real vs a nossa E12.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    zx = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))["proba"]
    zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
    zf = np.load(os.path.join(DATA_DIR, "nb08_e1_proba.npz"))["proba"]
    ze = np.load(os.path.join(DATA_DIR, "nb08_e4_proba.npz"))["proba"]
    Moof = np.hstack([zx, zh, zf, ze])
    E12c = {}
    for nome, C in [("logreg C=1.0", 1.0), ("logreg C=0.1", 0.1)]:
        f1s = []
        for tr, va in FOLDS:
            sc = StandardScaler()
            Ztr, Zva = sc.fit_transform(Xv[tr]), sc.transform(Xv[va])
            Atr = np.hstack([Ztr, Moof[tr]])
            Ava = np.hstack([Zva, Moof[va]])
            meta = LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
            meta.fit(Atr, y[tr])
            f1s.append(f1_score(y[va], meta.predict(Ava), average="macro"))
        E12c[nome] = f1s
        print(f"  {nome}: {np.mean(f1s):.4f}")
    gravar("e12c_metastack_full", E12c)
for k, v in E12c.items():
    registar("E12c", f"metastack+orig ({k})", v, base_f1s)
print(tabela(RESULTADOS[-len(E12c):]).to_string(index=False))


In [ ]:
E18 = carregar("e18_lgbm_tuned")
if E18 is None:
    # Tuning em 2 FASES (licao do t*): (1) halving com cv=2 interna escolhe params
    # (numero otimista, NAO conta); (2) params congelados correm a OOF externa de
    # 5 folds (este numero e o honesto). Tuning salta o rapido por desenho.
    import lightgbm as lgb
    from sklearn.experimental import enable_halving_search_cv
    from sklearn.model_selection import HalvingRandomSearchCV
    inner = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
    busca = HalvingRandomSearchCV(
        lgb.LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE,
                           verbose=-1, n_jobs=-1),
        {"n_estimators": [300, 500, 800], "learning_rate": [0.03, 0.05, 0.1],
         "num_leaves": [15, 31, 63], "min_child_samples": [20, 50, 100],
         "feature_fraction": [0.7, 0.8, 1.0], "lambda_l2": [0.0, 1.0, 10.0]},
        factor=2, cv=inner, scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1)
    busca.fit(Xv, y)
    params = {k: (float(v) if isinstance(v, float) else int(v))
              for k, v in busca.best_params_.items()}
    print("best:", params, round(float(busca.best_score_), 4))
    oof = np.empty(len(y), dtype=int)
    pro = np.zeros((len(y), K))
    f1s = []
    for tr, va in FOLDS:
        m = lgb.LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE,
                               verbose=-1, n_jobs=-1, **params)
        m.fit(Xv[tr], y[tr])
        oof[va] = m.predict(Xv[va])
        pro[va] = m.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], oof[va], average="macro"))
    E18 = {"f1": f1s, "params": params, "tune_score": round(float(busca.best_score_), 4)}
    np.savez(os.path.join(DATA_DIR, f"nb08_e18_lgbm_tuned{SUF}.npz"), oof=oof, proba=pro)
    gravar("e18_lgbm_tuned", E18)
registar("E18", "LGBM tunado (halving, congelado)", E18["f1"], base_f1s,
         nota="leaves=" + str(E18["params"]["num_leaves"]))
print(tabela(RESULTADOS[-1:]).to_string(index=False))


In [ ]:
E19 = carregar("e19_xgb_tuned")
if E19 is None:
    # XGB base NUNCA foi tunado (so defaults + alphas da E6): mesma disciplina 2 fases.
    from sklearn.experimental import enable_halving_search_cv
    from sklearn.model_selection import HalvingRandomSearchCV
    inner = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
    busca = HalvingRandomSearchCV(
        XGBClassifier(tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
                      eval_metric="mlogloss"),
        {"n_estimators": [300, 500, 800], "max_depth": [4, 6, 8],
         "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.7, 0.8, 1.0],
         "colsample_bytree": [0.7, 0.8, 1.0], "reg_lambda": [0.5, 1.0, 5.0]},
        factor=2, n_candidates=24, cv=inner, scoring="f1_macro",
        random_state=RANDOM_STATE, n_jobs=1)
    busca.fit(Xv, y, sample_weight=SW_BAL)
    params = {k: (float(v) if isinstance(v, float) else int(v))
              for k, v in busca.best_params_.items()}
    print("best:", params, round(float(busca.best_score_), 4))
    oof = np.empty(len(y), dtype=int)
    pro = np.zeros((len(y), K))
    f1s = []
    for tr, va in FOLDS:
        m = XGBClassifier(tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
                          eval_metric="mlogloss", **params)
        m.fit(Xv[tr], y[tr], sample_weight=SW_BAL[tr])
        oof[va] = m.predict(Xv[va])
        pro[va] = m.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], oof[va], average="macro"))
    E19 = {"f1": f1s, "params": params, "tune_score": round(float(busca.best_score_), 4)}
    np.savez(os.path.join(DATA_DIR, f"nb08_e19_xgb_tuned{SUF}.npz"), oof=oof, proba=pro)
    gravar("e19_xgb_tuned", E19)
registar("E19", "XGB tunado (halving, congelado)", E19["f1"], base_f1s,
         nota="depth=" + str(E19["params"]["max_depth"]))
print(tabela(RESULTADOS[-1:]).to_string(index=False))


In [ ]:
E20 = carregar("e20_catboost_tuned")
if E20 is None:
    # CatBoost perdeu 0.9pp com defaults: mesma disciplina 2 fases.
    from sklearn.experimental import enable_halving_search_cv
    from sklearn.model_selection import HalvingRandomSearchCV
    from catboost import CatBoostClassifier
    inner = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
    busca = HalvingRandomSearchCV(
        CatBoostClassifier(auto_class_weights="Balanced", random_seed=RANDOM_STATE,
                           thread_count=-1, verbose=False, allow_writing_files=False),
        {"iterations": [300, 500, 800], "depth": [4, 6, 8],
         "learning_rate": [0.03, 0.05, 0.1]},
        factor=2, cv=inner, scoring="f1_macro",
        random_state=RANDOM_STATE, n_jobs=1)
    busca.fit(Xv, y)
    params = {k: (float(v) if isinstance(v, float) else int(v))
              for k, v in busca.best_params_.items()}
    print("best:", params, round(float(busca.best_score_), 4))
    oof = np.empty(len(y), dtype=int)
    pro = np.zeros((len(y), K))
    f1s = []
    for tr, va in FOLDS:
        m = CatBoostClassifier(auto_class_weights="Balanced", random_seed=RANDOM_STATE,
                               thread_count=-1, verbose=False,
                               allow_writing_files=False, **params)
        m.fit(Xv[tr], y[tr])
        oof[va] = np.asarray(m.predict(Xv[va]), dtype=int).ravel()
        pro[va] = m.predict_proba(Xv[va])
        f1s.append(f1_score(y[va], oof[va], average="macro"))
    E20 = {"f1": f1s, "params": params, "tune_score": round(float(busca.best_score_), 4)}
    np.savez(os.path.join(DATA_DIR, f"nb08_e20_catboost_tuned{SUF}.npz"), oof=oof, proba=pro)
    gravar("e20_catboost_tuned", E20)
registar("E20", "CatBoost tunado (halving, congelado)", E20["f1"], base_f1s,
         nota="depth=" + str(E20["params"]["depth"]))
print(tabela(RESULTADOS[-1:]).to_string(index=False))


In [ ]:
E21 = carregar("e21_mi_top150")
if E21 is None:
    # Ranking MI IN-FOLD (top-150 do treino de cada fold, nunca do fold validado):
    # teste de PARIDADE para triagem rapida, nao para score (arvores ja ignoram lixo).
    from sklearn.feature_selection import mutual_info_classif
    f1s, tops = [], []
    for tr, va in FOLDS:
        mi = mutual_info_classif(Xv[tr], y[tr], random_state=RANDOM_STATE)
        top = np.argsort(mi)[-150:]
        tops.append(sorted(map(int, top)))
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
        m.fit(Xv[tr][:, top], y[tr], sample_weight=SW_BAL[tr])
        f1s.append(f1_score(y[va], m.predict(Xv[va][:, top]), average="macro"))
    inter = [len(set(tops[a]) & set(tops[b])) for a in range(5) for b in range(a+1, 5)]
    print("overlap medio top-150 entre folds:", round(float(sum(inter)/len(inter)), 1))
    E21 = {"f1": f1s}
    gravar("e21_mi_top150", E21)
registar("E21", "MI top-150 (in-fold, paridade)", E21["f1"], base_f1s)
print(tabela(RESULTADOS[-1:]).to_string(index=False))


In [ ]:
E22 = carregar("e22_kmeans")
if E22 is None:
    # KMeans como DIAGNOSTICO (clusters recuperam as faixas?) + FEATURES in-fold
    # (one-hot do cluster + distancia ao centroide). Scaling+KMeans ajustados SO
    # no treino de cada fold (sem leakage); o ARI full-data e so descritivo.
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import adjusted_rand_score
    zfull = StandardScaler().fit_transform(Xv)
    ari = adjusted_rand_score(y, KMeans(n_clusters=16, n_init=10,
                                        random_state=RANDOM_STATE).fit_predict(zfull))
    print("ARI (clusters vs faixas, full-data):", round(float(ari), 4))
    f1s = []
    for tr, va in FOLDS:
        sc = StandardScaler().fit(Xv[tr])
        km = KMeans(n_clusters=16, n_init=10, random_state=RANDOM_STATE).fit(sc.transform(Xv[tr]))
        ctr, cva = km.predict(sc.transform(Xv[tr])), km.predict(sc.transform(Xv[va]))
        dtr = km.transform(sc.transform(Xv[tr]))[np.arange(len(tr)), ctr][:, None]
        dva = km.transform(sc.transform(Xv[va]))[np.arange(len(va)), cva][:, None]
        otr = np.hstack([Xv[tr], (ctr[:, None] == np.arange(16)).astype(float), dtr])
        ova = np.hstack([Xv[va], (cva[:, None] == np.arange(16)).astype(float), dva])
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
        m.fit(otr, y[tr], sample_weight=SW_BAL[tr])
        f1s.append(f1_score(y[va], m.predict(ova), average="macro"))
    E22 = {"f1": f1s, "ari": round(float(ari), 4)}
    gravar("e22_kmeans", E22)
registar("E22", "KMeans-16 como features (in-fold)", E22["f1"], base_f1s,
         nota="ARI=" + str(E22["ari"]))
print(tabela(RESULTADOS[-1:]).to_string(index=False))


In [11]:
final = tabela(RESULTADOS)
print(final.to_string(index=False))
final.to_csv(os.path.join(DATA_DIR, f"nb08_resultados{SUF}.csv"), index=False)
print(f"\nGuardado em data/nb08_resultados{SUF}.csv")

exp                           variante     f1    std   delta  folds_ganhos                                    nota
 E8                 F&H + E4 (0.5/0.5) 0.4982 0.0093  0.0246             5                                        
 E8            F&H + pairwise (hesita) 0.4981 0.0100  0.0245             5                                        
 E1             Frank & Hall (mediana) 0.4979 0.0131  0.0243             5                                        
 E1            blend 0.5*F&H + 0.5*XGB 0.4973 0.0088  0.0237             5                                        
 E8                 F&H + E4 (0.7/0.3) 0.4970 0.0091  0.0234             5                                        
 E8               F&H + E4 + XGB (1/3) 0.4960 0.0087  0.0224             5                                        
 E1              Frank & Hall (argmax) 0.4935 0.0128  0.0199             5                                        
 E1            Frank & Hall (esperado) 0.4914 0.0089  0.0178             5      

## Feature engineering: primeira tentativa

A hipótese é que as 339 features estão certas mas incompletas, e que agregados novos podem acrescentar sinal. Testamos três coisas no mesmo protocolo. Primeiro, 27 variáveis construídas a partir de colunas já existentes: dimensão e composição do agregado familiar, doenças agrupadas por sistema, severidade de limitações, IMC categórico e utilização de cuidados. Segundo, um target encoding do estádio de vida, que combina situação profissional, estado civil, filhos e escolaridade numa só variável. Terceiro, a mesma variável usada como intervalo proibido.

A parte de LightGBM custa cerca de 20 minutos na primeira execução e fica guardada em `data/nb08_feature_eng.json`. Os restantes testes são rápidos e correm sempre.


In [ ]:
# Feature engineering: agregados, target encoding do estádio e máscaras.
import json
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

FEATS = list(X.columns)

# ---- variáveis construídas ----
main_cols = [c for c in FEATS if c.startswith("MAINSTAT_")]
mar_cols = [c for c in FEATS if c.startswith("MARSTALEGAL_")]
main = np.where(X[main_cols].to_numpy().sum(1) > 0, X[main_cols].to_numpy().argmax(1), len(main_cols))
mar = np.where(X[mar_cols].to_numpy().sum(1) > 0, X[mar_cols].to_numpy().argmax(1), len(mar_cols))
hat = X["HATLEVEL"].to_numpy()
hat_b = np.where(hat <= 2, 0, np.where(hat <= 4, 1, 2))
kids = X["HHNBPERS_0_13_COD"].to_numpy().astype(float)
hh = X["HHNBPERS_COD"].to_numpy().astype(float)
tem_filhos = (kids > 0).astype(int)
stage = np.array([f"{a}-{b}-{c}-{d}" for a, b, c, d in zip(main, mar, tem_filhos, hat_b)])
fx = X["FAIXA_MINIMA"].to_numpy().astype(int)

def _one(c):
    return X[c].to_numpy() if c in FEATS else np.zeros(len(X))

cardio = sum((X[c].to_numpy() == 1).astype(float) for c in ["CD1C", "CD1D", "CD1E", "CD1F"])
resp = sum((X[c].to_numpy() == 1).astype(float) for c in ["CD1A", "CD1B"])
msk = sum((X[c].to_numpy() == 1).astype(float) for c in ["CD1G", "CD1H", "CD1I"])
metab = sum((X[c].to_numpy() == 1).astype(float) for c in ["CD1J", "CD1L", "CD1N"])
pl = lambda c: np.clip(X[c].to_numpy().astype(float), 0, 3)
DET = np.column_stack([
    np.clip(hh - kids, 0, None), np.where(hh > 0, kids / np.maximum(hh, 1), 0),
    _one("HHTYPE_unipessoal") if "HHTYPE_unipessoal" in FEATS else np.zeros(len(X)),
    cardio, resp, msk, metab, (X["CD1O"].to_numpy() == 1).astype(float),
    np.where(pl("PL1") == 2, 1, pl("PL1")) + pl("PL2"),
    np.where(pl("PL3") == 2, 1, pl("PL3")) + pl("PL4"),
    pl("PL5") + pl("PL6"), pl("PL7") + pl("PL8"),
    _one("IMC"), _one("AM3"),
    np.isin(X["PA1_ESTADO"].to_numpy(), [0, 1]).astype(float),
])
DET = np.nan_to_num(DET, nan=-1.0, posinf=-1.0, neginf=-1.0)

# ---- target encoding do estádio, cross-fitted ----
def te_cross(tr, va, alpha=20.0):
    keys, yy = stage[tr], y[tr]
    prior = yy.mean()
    agg = pd.DataFrame({"k": keys, "y": yy}).groupby("k")["y"].agg(["sum", "count"])
    s = agg.loc[keys, "sum"].to_numpy() - yy
    c = agg.loc[keys, "count"].to_numpy() - 1
    te_tr = (s + alpha * prior) / (c + alpha)
    kv = stage[va]
    te_va = ((agg["sum"].reindex(kv).fillna(0).to_numpy() + alpha * prior) /
             (agg["count"].reindex(kv).fillna(0).to_numpy() + alpha))
    return te_tr[:, None], te_va[:, None]

te_all = np.zeros((len(y), 1))
for tr, va in FOLDS:
    te_all[va] = te_cross(tr, va)[1]
corr = float(np.corrcoef(te_all[:, 0], y)[0, 1])
mae = float(np.abs(te_all[:, 0] - y).mean())
print(f"target encoding do estádio: correlação {corr:.3f}, erro absoluto médio {mae:.2f} faixas")

# ---- resultados LightGBM guardados ----
FE_PATH = os.path.join(DATA_DIR, "nb08_feature_eng.json")
FE = json.load(open(FE_PATH)) if os.path.exists(FE_PATH) else {}

def _linha(nome, d, base):
    return {"variante": nome, "f1": round(d["mean"], 4),
            "std": round(d.get("std", 0), 4), "delta": round(d["mean"] - base, 4)}

if "lgbm_base" in FE:
    b = FE["lgbm_base"]["mean"]
    tabela_fe = pd.DataFrame([
        _linha("LightGBM base (339 features)", FE["lgbm_base"], b),
        _linha("LightGBM + 27 agregados", FE["lgbm_agregados"], b),
        _linha("LightGBM + agregados + estádio TE", FE["lgbm_agregados_te"], b),
        _linha("LightGBM feature_fraction=1.0", FE["lgbm_ff10"], b),
    ])
    print("\nParte LightGBM (checkpoint data/nb08_feature_eng.json)")
    print(tabela_fe.to_string(index=False))
    for nome, key, bkey in [("XGB base (339 features)", "xgb_base", "xgb_base"),
                            ("XGB + 27 agregados", "xgb_det", "xgb_base"),
                            ("XGB + agregados + estádio TE", "xgb_det_te", "xgb_base"),
                            ("Frank & Hall base (339 features)", "fh_base", "fh_base"),
                            ("Frank & Hall + 27 agregados", "fh_det", "fh_base"),
                            ("Frank & Hall + agregados + estádio TE", "fh_det_te", "fh_base")]:
        if key in FE and bkey in FE:
            d = FE[key]["mean"] - FE[bkey]["mean"]
            print(f"{nome}: {FE[key]['mean']:.4f}  delta {d:+.4f}")
    if "residual_lgbm" in FE:
        print("\nEstádio como residual: o modelo prevê a desvio dentro do grupo")
        print(f"round(estádio) sozinho: {FE['round_estadio']['mean']:.4f}")
        for nome, key in [("residual LGBM", "residual_lgbm"),
                          ("residual XGB mse", "residual_xgb_mse"),
                          ("residual XGB mae", "residual_xgb_mae")]:
            d = FE[key]
            print(f"{nome}: {d['mean']:.4f}  delta vs F&H "
                  f"{d['mean'] - FE['fh_base']['mean']:+.4f}")
else:
    print("\ncheckpoint data/nb08_feature_eng.json em falta: a parte LightGBM não está guardada")

# ---- meta + estádio TE ----
zx = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))["proba"]
zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
zf = np.load(os.path.join(DATA_DIR, "nb08_e1_proba.npz"))["proba"]
ze = np.load(os.path.join(DATA_DIR, "nb08_e4_proba.npz"))["proba"]
META = np.hstack([zx, zh, zf, ze])

def meta_f1(com_te):
    f1s = []
    for tr, va in FOLDS:
        A, B = te_cross(tr, va) if com_te else (np.zeros((len(tr), 0)), np.zeros((len(va), 0)))
        m = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        m.fit(np.hstack([META[tr], A]), y[tr])
        f1s.append(f1_score(y[va], m.predict(np.hstack([META[va], B])), average="macro"))
    return np.array(f1s)

mb = meta_f1(False)
mt = meta_f1(True)
print(f"\nmeta4 base: {mb.mean():.4f} | meta4 + estádio TE: {mt.mean():.4f} "
      f"(delta {mt.mean() - mb.mean():+.4f}, {int((mt > mb).sum())}/5 folds)")

# ---- máscaras de intervalo ----
proba_meta = np.zeros_like(zx)
for tr, va in FOLDS:
    m = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
    m.fit(META[tr], y[tr])
    proba_meta[va] = m.predict_proba(META[va])

def mascara(qlo, qhi, minimo):
    f1s = []
    for tr, va in FOLDS:
        d = pd.DataFrame({"k": stage[tr], "y": y[tr]}).groupby("k")["y"]
        q = d.quantile([qlo, qhi]).unstack()
        ok = d.count()
        ok = ok[ok >= minimo].index
        keys = stage[va]
        lo = pd.Series(keys).map(q.loc[ok, qlo]).fillna(-1).to_numpy()
        hi = pd.Series(keys).map(q.loc[ok, qhi]).fillna(99).to_numpy()
        p = proba_meta[va].copy()
        for k in range(p.shape[1]):
            p[:, k] = np.where((k < lo) | (k > hi), -1.0, p[:, k])
        f1s.append(f1_score(y[va], p.argmax(1), average="macro"))
    return np.array(f1s)

base = float(f1_score(y, proba_meta.argmax(1), average="macro"))
print(f"\nmeta4 sem máscara: {base:.4f}")
for nome, args in [("estádio q[0.10,0.90]", (0.10, 0.90, 30)),
                   ("estádio q[0.05,0.95]", (0.05, 0.95, 30))]:
    f = mascara(*args)
    print(f"{nome}: {f.mean():.4f} (delta {f.mean() - base:+.4f})")

p = proba_meta.copy()
for k in range(p.shape[1]):
    p[:, k] = np.where(k < fx, -1.0, p[:, k])
print(f"máscara FAIXA_MINIMA: {f1_score(y, p.argmax(1), average='macro'):.4f}")


target encoding do estádio: correlação 0.877, erro absoluto médio 1.44 faixas

Parte LightGBM (checkpoint data/nb08_feature_eng.json)
                         variante     f1    std   delta
     LightGBM base (339 features) 0.4743 0.0070  0.0000
          LightGBM + 27 agregados 0.4674 0.0084 -0.0069
LightGBM + agregados + estádio TE 0.3121 0.0149 -0.1622
    LightGBM feature_fraction=1.0 0.4662 0.0081 -0.0081
XGB base (339 features): 0.4736  delta +0.0000
XGB + 27 agregados: 0.4756  delta +0.0020
XGB + agregados + estádio TE: 0.3149  delta -0.1587
Frank & Hall base (339 features): 0.4979  delta +0.0000
Frank & Hall + 27 agregados: 0.4676  delta -0.0303
Frank & Hall + agregados + estádio TE: 0.3499  delta -0.1480

Estádio como residual: o modelo prevê a desvio dentro do grupo
round(estádio) sozinho: 0.1510
residual LGBM: 0.3317  delta vs F&H -0.1662
residual XGB mse: 0.2977  delta vs F&H -0.2002
residual XGB mae: 0.2844  delta vs F&H -0.2135

meta4 base: 0.5037 | meta4 + estádio TE: 0.

### O que se aprendeu

A tentativa falhou, e as três formas de falhar são informativas.

| Variante | F1 | Diferença | Veredito |
|---|---:|---:|---|
| Variante | F1 | Diferença | Veredito |
|---|---:|---:|---|
| LightGBM base, 339 features | 0.4743 | 0.0000 | referência |
| LightGBM + 27 agregados | 0.4674 | -0.0069 | rejeitado, 1/5 folds |
| LightGBM + agregados + estádio TE | 0.3121 | -0.1622 | colapso |
| LightGBM com feature_fraction 1.0 | 0.4662 | -0.0081 | pior do que 0.8 |
| XGB base, 339 features | 0.4736 | 0.0000 | referência |
| XGB + 27 agregados | 0.4756 | +0.0020 | ruído, 2/5 folds |
| XGB + agregados + estádio TE | 0.3149 | -0.1587 | colapso |
| Frank & Hall base, 339 features | 0.4979 | 0.0000 | referência |
| Frank & Hall + 27 agregados | 0.4676 | -0.0303 | rejeitado, 0/5 folds |
| Frank & Hall + agregados + estádio TE | 0.3499 | -0.1480 | colapso |
| meta4 + estádio TE | 0.5022 | -0.0015 | rejeitado, 3/5 folds |
| meta4 com máscara do estádio q[0.10,0.90] | 0.4752 | -0.0287 | rejeitado |
| meta4 com máscara do estádio q[0.05,0.95] | 0.4941 | -0.0098 | rejeitado |
| meta4 com máscara FAIXA_MINIMA | 0.5044 | +0.0005 | neutro |
| round(estádio) sozinho | 0.1510 | -0.3469 | só o valor do grupo |
| residual LGBM, desvio dentro do grupo | 0.3317 | -0.1662 | rejeitado, 0/5 vs XGB |
| residual XGB, erro médio quadrático | 0.2977 | -0.2002 | rejeitado |
| residual XGB, erro médio absoluto | 0.2844 | -0.2135 | rejeitado |

- O target encoding do estádio é a variável mais próxima da idade que já construímos: correlação 0.877 com a classe real e erro absoluto médio de 1.44 faixas. Ainda assim, no LightGBM derruba o F1 de 0.4743 para 0.3121. A árvore passa a decidir pelo valor esperado do grupo e as previsões colapsam para o centro, o que mata as classes extremas que o Macro-F1 pesa igualmente.
- A máscara por estádio faz o mesmo por outro caminho. Proibir classes fora dos quantis do grupo corta a resposta verdadeira nos grupos pequenos, e mesmo a versão larga de q[0.05,0.95] custa 0.0098.
- Os 27 agregados são redundantes com as colunas que já existem. Ainda por cima, o LightGBM usa `feature_fraction=0.8`, e acrescentar colunas dilui a sub-amostra disponível em cada split. O controlo com `feature_fraction=1.0` dá 0.4662, pior do que 0.8 em 0.0081, o que confirma que a sub-amostragem de features é a regularização que nos está a dar o 0.4743.
- O padrão repete-se nas três famílias de modelo: XGB ganha 0.0020, que é ruído com 2/5 folds, LightGBM perde 0.0069 e Frank & Hall perde 0.0303 em todas as folds. A porta dos agregados fecha-se nos três.
- O estádio falha nos quatro formatos testados: como variável numérica no classificador, como variável do meta, como intervalo proibido e como alvo de regressão. O residual é a versão mais fraca de todas, com 0.3317 no LightGBM, porque arredondar o estádio sozinho já dá apenas 0.1510: o valor do grupo está muito comprimido para o centro e nunca atinge as classes extremas.
- O limite inferior `FAIXA_MINIMA` continua neutro-positivo, com 0.0005, mas não mexe a agulha.

**Lições para a próxima ronda.** Agregados de colunas que o modelo já vê não acrescentam. Qualquer transformação que converta a tarefa em regressão para a média destrói o Macro-F1. Features novas têm custo de diluição, por isso, se acrescentarmos colunas, temos de subir o `feature_fraction` ou fazer seleção.

**Próximos testes, por ordem.** Primeiro, categóricas nativas no CatBoost e no LightGBM a partir do texto do NB01, que nunca foram testadas e não passam por agregados. Segundo, novos proxies de idade, com `SK2` e as variáveis de tabaco que não entram no `IDADE_MINIMA`. Terceiro, alargar o `IDADE_MINIMA` para limites superiores derivados do routing. O teste do residual já foi feito e rejeitado.


## Diagnóstico do melhor modelo

Esta secção analisa o metastack de quatro bases, o melhor resultado deste notebook, com 0.5037 de Macro-F1 out-of-fold. As probabilidades das quatro bases já estão guardadas, por isso a análise não treina modelos base. A célula seguinte reconstrói o meta-modelo e imprime a matriz de confusão, o F1 por classe, a calibração e os testes de erro. A leitura dos números e as experiências seguintes estão nos blocos de texto abaixo.


In [ ]:
# Reconstrução do metastack de quatro bases, sem treinar as bases.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             log_loss, precision_recall_fscore_support)
from scipy.stats import binomtest

zx = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))["proba"]
zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
zf = np.load(os.path.join(DATA_DIR, f"nb08_e1_proba{SUF}.npz"))["proba"]
ze = np.load(os.path.join(DATA_DIR, f"nb08_e4_proba{SUF}.npz"))["proba"]
M = np.hstack([zx, zh, zf, ze])

proba_meta = np.zeros_like(zx)
for tr, va in FOLDS:
    meta = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
    meta.fit(M[tr], y[tr])
    proba_meta[va] = meta.predict_proba(M[va])
pred_meta = proba_meta.argmax(axis=1)

print(f"accuracy {accuracy_score(y, pred_meta):.4f} "
      f"| macro F1 {f1_score(y, pred_meta, average='macro'):.4f} "
      f"| weighted F1 {f1_score(y, pred_meta, average='weighted'):.4f}")
err = np.abs(pred_meta - y)
print(f"off-by-one {(err == 1).mean():.2%} | |err| >= 3 {(err >= 3).mean():.2%} "
      f"| mean |err| {err.mean():.3f}")

p, r, f, n = precision_recall_fscore_support(y, pred_meta, labels=CLASSES, zero_division=0)
print(pd.DataFrame({"classe": CLASSES, "n": n, "precisao": p.round(3),
                    "recall": r.round(3), "f1": f.round(3)}).to_string(index=False))

cm = confusion_matrix(y, pred_meta, labels=CLASSES)
print("\nMatriz de confusao, linhas verdade e colunas previsao")
print(pd.DataFrame(cm, index=[f"t{i}" for i in CLASSES],
                   columns=[f"p{i}" for i in CLASSES]).to_string())

conf = proba_meta.max(axis=1)

def _ece(y_true, proba, bins=10):
    c = proba.max(axis=1)
    a = (proba.argmax(axis=1) == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for i in range(bins):
        m = (c > edges[i]) & (c <= edges[i + 1])
        if m.sum():
            e += m.mean() * abs(a[m].mean() - c[m].mean())
    return e

print(f"\nNLL {log_loss(y, proba_meta):.4f} | ECE {_ece(y, proba_meta):.4f} "
      f"| confianca media {conf.mean():.4f}")
for t in (0.6, 0.8, 0.95):
    m = conf >= t
    print(f"p >= {t}: {int(m.sum())} linhas, {int(((pred_meta != y) & m).sum())} erradas "
          f"({((pred_meta != y) & m).sum() / max(m.sum(), 1):.1%})")

off = sorted(((i, j, cm[i, j]) for i in CLASSES for j in CLASSES if i != j),
             key=lambda t: -t[2])[:10]
print("\nTop 10 erros, verdade -> previsao")
for i, j, c in off:
    print(f"  {i} -> {j}: {c} ({c / n[i]:.1%} da classe)")

def _mcnemar(a, b):
    meta_right = int(((a == y) & (b != y)).sum())
    other_right = int(((a != y) & (b == y)).sum())
    return meta_right, other_right, binomtest(meta_right, meta_right + other_right, 0.5).pvalue

for nome, pm in [("XGB", zx.argmax(1)), ("F&H", zf.argmax(1)), ("hierarquia", ze.argmax(1))]:
    mr, or_, pv = _mcnemar(pred_meta, pm)
    print(f"McNemar meta vs {nome}: meta certo {mr}, outro certo {or_}, p {pv:.2e}")

fx = train["FAIXA_MINIMA"].to_numpy()
for nome, m in [("com proxy", fx > 0), ("sem proxy", fx == 0)]:
    print(f"{nome}: {int(m.sum())} linhas, accuracy {(pred_meta[m] == y[m]).mean():.4f}, "
          f"macro F1 {f1_score(y[m], pred_meta[m], average='macro'):.4f}")


accuracy 0.5016 | macro F1 0.5039 | weighted F1 0.5004
off-by-one 38.12% | |err| >= 3 2.85% | mean |err| 0.655
 classe    n  precisao  recall    f1
      0  207     0.824   0.928 0.873
      1  172     0.459   0.494 0.476
      2  399     0.620   0.571 0.595
      3  377     0.410   0.483 0.443
      4  449     0.318   0.350 0.333
      5  647     0.382   0.399 0.390
      6  859     0.396   0.360 0.377
      7  862     0.426   0.413 0.420
      8  905     0.692   0.663 0.677
      9 1014     0.683   0.684 0.684
     10 1062     0.656   0.682 0.669
     11 1040     0.581   0.563 0.572
     12 1000     0.404   0.380 0.392
     13  823     0.315   0.272 0.292
     14  650     0.348   0.375 0.361
     15  496     0.466   0.562 0.510

Matriz de confusao, linhas verdade e colunas previsao
      p0  p1   p2   p3   p4   p5   p6   p7   p8   p9  p10  p11  p12  p13  p14  p15
t0   192  11    0    3    0    0    0    1    0    0    0    0    0    0    0    0
t1    35  85   49    3    0    0    0  

### O que os números dizem

- O metastack acerta 50.2% das linhas e tem 0.5039 de Macro-F1 global. A média das cinco folds é 0.5037.
- 76.5% dos erros são por uma faixa e 59.1% estão em 15 pares adjacentes. As quatro maiores células, 9 para 10, 10 para 9, 11 para 12 e 12 para 11, somam 20.6% de todos os erros.
- As piores classes são 13 (75-79, F1 0.292), 4 (30-34, 0.333), 6 (40-44, 0.377), 5 (35-39, 0.390) e 12 (70-74, 0.392). As melhores são 0 (15-17, 0.873), 9 (55-59, 0.684), 8 (50-54, 0.677) e 10 (60-64, 0.669).
- A fronteira 50-54 para 55-59 está resolvida. Apenas 4 linhas em 10 962 atravessam essa fronteira no sentido jovem para velho e nenhuma no sentido inverso. As variáveis de routing do bloco 55+ fazem essa separação.
- A confiança está bem calibrada, com ECE 0.021 e confiança média 0.489 contra acerto 0.502. O problema não é a calibração.
- 26.5% das previsões com p >= 0.6 estão erradas e 8.8% das com p >= 0.8 também. Não existe uma zona de confiança alta utilizável para pseudo-labels.
- O modelo está sobreconfiante nas classes 1 e 13, duas das piores, e subconfiante nas classes 0, 8, 11 e 15.
- Em 42.9% dos erros as quatro bases concordam e todas erram. Em 66.7% dos erros nenhuma base acerta. Um seletor perfeito entre as bases daria 0.667 de accuracy, por isso o teto do stacking é baixo.
- O teste de McNemar confirma que o ganho sobre XGB, Frank & Hall e hierarquia é real, mas o meta vence apenas 54% a 56% das linhas em que discorda das bases.


### Testes baratos já feitos

Todos usam as mesmas folds e comparam com 0.5037.

| Teste | F1 | Diferença | Leitura |
|---|---:|---:|---|
| meta com FAIXA_MINIMA e N_SEM_RESPOSTA | 0.5030 | -0.0007 | rejeitado |
| meta com regressão logística e ExtraTrees como bases extra | 0.5039 | +0.0002 | ruído |
| meta não linear com HGB | 0.4579 | -0.0458 | rejeitado |
| bias por classe cross-fitted | 0.5029 | -0.0008 | rejeitado |
| média com Frank & Hall ou hierarquia | 0.4927 a 0.5010 | -0.011 a -0.003 | rejeitado |
| FAIXA_MINIMA em one-hot | 0.5044 | +0.0007 | 2/5 folds, ruído |
| bases calibradas por temperatura antes do meta | 0.5011 | -0.0026 | rejeitado |
| grelha de C entre 0.02 e 0.3 | 0.5020 a 0.5037 | menor ou igual a zero | C=0.1 continua ótimo |

O metastack está num ótimo local. Nenhum pós-processamento, peso, calibração ou blend melhora de forma fiável.


### O que os dados dizem

- 44.4% das linhas têm um proxy de idade, FAIXA_MINIMA maior do que 0. Nestas linhas o erro médio absoluto é 0.607 contra 0.692 nas restantes, mas o Macro-F1 do subgrupo é menor, 0.461 contra 0.485, porque a composição etária é diferente.
- O proxy é um piso, não um ponto. Só 7.7% das linhas com proxy têm a classe igual ao piso e a mediana da distância é 4 faixas.
- N_SEM_RESPOSTA tem correlação 0.029 com o erro e 0.071 com o erro absoluto. É estatisticamente significativo com 10 962 linhas, mas praticamente irrelevante.
- A importância do LightGBM está distribuída: socio 27.1%, saúde 21.7%, routing e idade 17.0%, outras 34.1%. As 17 flags de routing do bloco 55+ somam 16.8% do ganho. Somando as perguntas reprodutivas, cerca de 19.3% do ganho vem do routing do questionário.
- As features mais importantes são MAINSTAT aluno (8.33%), MAINSTAT reformado (6.10%), IN15 (4.56%), MARSTALEGAL solteiro (4.31%) e HATLEVEL (3.36%).
- O bloco 55+ está separado por routing, não por idade. Dentro de cada bloco, o modelo usa estádio de vida e saúde. As classes do meio dos blocos, 30-49 e 65-79, são as que têm menos sinal.


### Onde investir a seguir

O cenário realista é ganhar 0.005 a 0.02. Chegar a 0.55 exige uma variável de idade mais fina ou dados externos, não mais uma camada de stacking.

| # | Experiência | Tipo | Ganho esperado | Custo | Repetição |
|---:|---|---|---:|---|---|
| 1 | Target encoding de estádios de vida, com MAINSTAT, MARSTALEGAL, HATLEVEL e filhos | dados | +0.004 a +0.010 | 1 a 2 dias | nova |
| 2 | Categóricas nativas no CatBoost e LightGBM, a partir do texto do NB01 | dados | +0.003 a +0.010 | 1 a 2 dias | repetição dos encodings do NB05, agora com modelos novos |
| 3 | Intervalos de idade a partir do routing: mínimo, máximo e máscara dura | dados | +0.003 a +0.012 | 2 a 3 dias | nova |
| 4 | Novos proxies de idade, incluindo SK2 e IN24, IN26, IN45 e IN47 | dados | +0.002 a +0.006 | 1 dia | nova |
| 5 | Seleção de features por ganho ou permutation importance | dados | +0.002 a +0.008 | 1 dia | nova |
| 6 | Modelo ordinal latente, CORAL ou logit ordinal, como base do meta | modelo | +0.003 a +0.008 | 2 dias | repetição do Frank & Hall e dos alvos suaves |
| 7 | Bases por bloco de features: routing, socio, saúde | modelo | +0.002 a +0.006 | 1 a 2 dias | nova |
| 8 | Repetir os blocos de contagens e clínicos no LightGBM | dados | 0 a +0.005 | 0.5 dia | repetição do NB04, que testou só HGB |
| 9 | Hierarquia de dois blocos, 0-8 e 9-15 | modelo | +0.002 a +0.006 | 2 dias | repetição da hierarquia atual, com baixa confiança |
| 10 | Pseudo-labeling do teste com p >= 0.8 | dados | +0.001 a +0.004 | 0.5 dia | nova |

Não repetir: score de regressão como meta-feature (-0.0044), meta não linear (-0.0458), blends e thresholds. Todos foram testados e pioraram.

A ordem recomendada é 1, 3, 6, 2, 4, 5. A experiência 3 tem o maior teto, mas depende de a auditoria do routing encontrar mais filtros do que o corte dos 55 anos.


## Submissão

Treinamos os 15 classificadores cumulativos em todo o treino e prevemos o teste. O ficheiro `sub07_frank_hall.csv` usa a mediana da distribuição, com 0.4979 na validação. O ficheiro `sub07b_fh_xgb_blend.csv` usa a média com o XGBoost, com 0.4973, e tem menor variância entre folds.

Aplicamos o limite `FAIXA_MINIMA` quando a coluna existe, como no Notebook 03. Os dois ficheiros ficam em `submissions/`. Nada é submetido automaticamente.


In [12]:
SUB_DIR = os.path.join(os.path.dirname(DATA_DIR), "submissions")
CACHE = os.path.join(DATA_DIR, f"nb08_sub07_pred{SUF}.npz")
test = pd.read_csv(os.path.join(DATA_DIR, "test_nb02.csv"))
Xte = test[X.columns].to_numpy()

if os.path.exists(CACHE):
    z = np.load(CACHE)
    pred_fh, pred_blend = z["pred_fh"], z["pred_blend"]
    print("submissao carregada do checkpoint")
else:
    cum_te = np.ones((len(test), K + 1))
    for k in range(1, K):
        yk = (y >= k).astype(int)
        m = XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
            random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss",
        )
        m.fit(Xv, yk, sample_weight=compute_sample_weight("balanced", yk))
        cum_te[:, k] = m.predict_proba(Xte)[:, 1]
    cum_te[:, 0], cum_te[:, K] = 1.0, 0.0
    cum_te = np.minimum.accumulate(cum_te, axis=1)
    p_te = np.clip(cum_te[:, :K] - cum_te[:, 1:], 0, None)
    p_te = p_te / p_te.sum(axis=1, keepdims=True)
    pred_fh = (np.cumsum(p_te, axis=1) < 0.5).sum(axis=1)

    mx = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=4, eval_metric="mlogloss",
    )
    mx.fit(Xv, y, sample_weight=SW_BAL)
    pred_blend = (0.5 * p_te + 0.5 * mx.predict_proba(Xte)).argmax(axis=1)

    if "FAIXA_MINIMA" in test.columns:
        pred_fh = np.maximum(pred_fh, test["FAIXA_MINIMA"].to_numpy().astype(int))
        pred_blend = np.maximum(pred_blend, test["FAIXA_MINIMA"].to_numpy().astype(int))
    np.savez(CACHE, pred_fh=pred_fh, pred_blend=pred_blend)
    print("submissao treinada agora")

base_sub = pd.read_csv(os.path.join(DATA_DIR, "example_submission_10.csv"))
for nome, pred in [("sub07_frank_hall.csv", pred_fh),
                   ("sub07b_fh_xgb_blend.csv", pred_blend)]:
    sub = base_sub.copy()
    sub[TARGET] = pred
    sub.to_csv(os.path.join(SUB_DIR, nome), index=False)
    print(f"{nome}: {len(sub)} linhas | distribuicao {sub[TARGET].value_counts().sort_index().to_dict()}")
print("Submeter MANUALMENTE; o notebook nunca submete.")

submissao treinada agora
sub07_frank_hall.csv: 3655 linhas | distribuicao {0: 71, 1: 58, 2: 133, 3: 151, 4: 176, 5: 246, 6: 302, 7: 299, 8: 191, 9: 303, 10: 398, 11: 204, 12: 360, 13: 333, 14: 279, 15: 151}
sub07b_fh_xgb_blend.csv: 3655 linhas | distribuicao {0: 71, 1: 51, 2: 143, 3: 146, 4: 179, 5: 218, 6: 275, 7: 297, 8: 247, 9: 333, 10: 379, 11: 266, 12: 356, 13: 289, 14: 236, 15: 169}
Submeter MANUALMENTE; o notebook nunca submete.


## Submissão do meta-stacking

Esta célula treina as quatro bases em todo o treino: XGBoost, HGB, decomposição cumulativa e hierarquia. Depois gera as probabilidades do teste e aplica o meta-modelo de regressão logística com C igual a 0.1. O resultado fica em `sub08_metastack.csv`.

Ao contrário da submissão anterior, não aplica a correção `FAIXA_MINIMA`, para reproduzir exatamente a validação out-of-fold que deu 0.5037.


In [ ]:
# --- SUB08: meta-stacking com regressão logística C=0.1, treinado em todo o treino ---
# Replica fiel da OOF validada (0.5037): SEM correcao de FAIXA_MINIMA (a OOF nao tem).
SUB8_DIR = os.path.join(os.path.dirname(DATA_DIR), "submissions")
SUB8_CACHE = os.path.join(DATA_DIR, "nb08_sub08_probas.npz")
SUB8_CSV = os.path.join(SUB8_DIR, "sub08_metastack.csv")
te8 = pd.read_csv(os.path.join(DATA_DIR, "test_nb02.csv"))
Xte_df = te8[X.columns]
Xte = Xte_df.to_numpy()

if os.path.exists(SUB8_CACHE):
    z8 = np.load(SUB8_CACHE)
    M_te, pred_te = z8["M_te"], z8["pred_te"]
    print("teste carregado do checkpoint")
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    # 1) XGB multiclasse full-train
    mx = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                       colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                       random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
    mx.fit(Xv, y, sample_weight=SW_BAL)
    p_x = mx.predict_proba(Xte)
    print("XGB teste: ok")
    # 2) HGB full-train (mesmo da receita NB07)
    mh = HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE)
    mh.fit(X, y)
    p_h = mh.predict_proba(Xte_df)
    print("HGB teste: ok")
    # 3) F&H 15 binarios full-train
    cum = np.ones((len(te8), K + 1))
    for k in range(1, K):
        yk = (y >= k).astype(int)
        m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                          random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
        m.fit(Xv, yk, sample_weight=compute_sample_weight("balanced", yk))
        cum[:, k] = m.predict_proba(Xte)[:, 1]
    cum[:, 0], cum[:, K] = 1.0, 0.0
    cum = np.minimum.accumulate(cum, axis=1)
    p_f = np.clip(cum[:, :K] - cum[:, 1:], 0, None)
    p_f = p_f / p_f.sum(axis=1, keepdims=True)
    print("F&H teste: ok")
    # 4) hierarquia grosso-fino treinada em todo o treino (mesmos grupos)
    GRUPOS8 = {0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8], 2: [9, 10, 11], 3: [12, 13, 14, 15]}
    gfull = np.full(len(y), -1, dtype=int)
    for gi, classes in GRUPOS8.items():
        gfull[np.isin(y, classes)] = gi
    m1 = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                       colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                       random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
    m1.fit(Xv, gfull, sample_weight=compute_sample_weight("balanced", gfull))
    pg_te = m1.predict_proba(Xte)
    p_e = np.zeros((len(te8), K))
    for gi, classes in GRUPOS8.items():
        sel = np.isin(y, classes)
        remap = {c: i for i, c in enumerate(classes)}
        y_enc = np.array([remap[v] for v in y[sel]])
        m2 = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                           colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                           random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss")
        m2.fit(Xv[sel], y_enc, sample_weight=compute_sample_weight("balanced", y_enc))
        p_e[:, classes] = pg_te[:, [gi]] * m2.predict_proba(Xte)
    print("hierarquia teste: ok")
    M_te = np.hstack([p_x, p_h, p_f, p_e])
    # 5) meta-modelo treinado na previsão out-of-fold completa
    zx = np.load(os.path.join(DATA_DIR, "xgb_oof.npz"))["proba"]
    zh = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))["proba"]
    zf = np.load(os.path.join(DATA_DIR, "nb08_e1_proba.npz"))["proba"]
    ze = np.load(os.path.join(DATA_DIR, "nb08_e4_proba.npz"))["proba"]
    meta = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
    meta.fit(np.hstack([zx, zh, zf, ze]), y)
    pred_te = meta.predict(M_te)
    np.savez(SUB8_CACHE, M_te=M_te, pred_te=pred_te)
    print("teste calculado e guardado")

base_sub = pd.read_csv(os.path.join(DATA_DIR, "example_submission_10.csv"))
assert len(base_sub) == len(pred_te) == 3655
sub = base_sub.copy()
sub[TARGET] = pred_te
sub.to_csv(SUB8_CSV, index=False)
print("sub08_metastack.csv gravada:", sub[TARGET].value_counts().sort_index().to_dict())
print("Submeter MANUALMENTE; o notebook nunca submete.")
